# 05 - MNIST Classifier: Putting It All Together

**Goal:** Build a complete image classifier from scratch.

---

## MNIST Dataset

The "Hello World" of deep learning:
- 70,000 grayscale images of handwritten digits (0-9)
- Each image is 28x28 pixels
- 60,000 training + 10,000 test images

Task: Look at an image → predict which digit it is.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 1: Load the Data

In [ ]:
# Transforms: convert to tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert PIL image to tensor (0-255 -> 0-1)
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Download and load data
train_dataset = datasets.MNIST(
    root='/workspace/data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='/workspace/data',
    train=False,
    download=True,
    transform=transform
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Look at a sample
image, label = train_dataset[0]

print(f"Image shape: {image.shape}")  # (1, 28, 28) = channels, height, width
print(f"Label: {label}")

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap='gray')  # squeeze removes channel dim
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Create DataLoaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

## Step 2: Define the Model

We'll build a CNN with:
- 2 convolutional layers (feature extraction)
- 2 fully connected layers (classification)

In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 14x14 -> 14x14
        
        self.pool = nn.MaxPool2d(2, 2)  # Halves dimensions
        self.dropout = nn.Dropout(0.25)
        
        # After conv1+pool: 28x28 -> 14x14
        # After conv2+pool: 14x14 -> 7x7
        # Flatten: 64 * 7 * 7 = 3136
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)  # 10 digit classes
        
    def forward(self, x):
        # x: (batch, 1, 28, 28)
        
        # Conv block 1
        x = self.pool(torch.relu(self.conv1(x)))  # -> (batch, 32, 14, 14)
        
        # Conv block 2
        x = self.pool(torch.relu(self.conv2(x)))  # -> (batch, 64, 7, 7)
        x = self.dropout(x)
        
        # Flatten
        x = x.view(x.size(0), -1)  # -> (batch, 3136)
        
        # Fully connected
        x = torch.relu(self.fc1(x))  # -> (batch, 128)
        x = self.dropout(x)
        x = self.fc2(x)  # -> (batch, 10)
        
        return x

# Create model
model = MNISTClassifier().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test forward pass
sample_batch = torch.randn(4, 1, 28, 28).to(device)
output = model(sample_batch)
print(f"Input: {sample_batch.shape}")
print(f"Output: {output.shape}")
print(f"Output (logits for first sample): {output[0].detach().cpu().numpy()}")

## Step 3: Setup Training

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training settings
num_epochs = 5

## Step 4: Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Stats
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

In [ ]:
# Train!
train_losses = []
test_losses = []
train_accs = []
test_accs = []

print("Starting training...\n")

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Evaluate
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Test Loss:  {test_loss:.4f}, Test Acc:  {test_acc:.2f}%")
    print()

print(f"Final Test Accuracy: {test_accs[-1]:.2f}%")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(train_losses, label='Train')
ax1.plot(test_losses, label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Over Training')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(train_accs, label='Train')
ax2.plot(test_accs, label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Over Training')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Make Predictions

In [ ]:
# Get some test images
model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

# Predict
with torch.no_grad():
    outputs = model(images)
    probabilities = torch.softmax(outputs, dim=1)
    predictions = outputs.argmax(dim=1)

# Show results
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = images[i].cpu().squeeze()
    pred = predictions[i].item()
    true = labels[i].item()
    prob = probabilities[i, pred].item()
    
    ax.imshow(img, cmap='gray')
    color = 'green' if pred == true else 'red'
    ax.set_title(f'Pred: {pred} ({prob:.1%})\nTrue: {true}', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 6: Save the Model

In [ ]:
# Save model weights
torch.save(model.state_dict(), '/workspace/data/mnist_model.pth')
print("Model saved!")

# To load later:
# new_model = MNISTClassifier()
# new_model.load_state_dict(torch.load('/workspace/data/mnist_model.pth'))
# new_model.eval()

## What We Built

A complete deep learning pipeline:

```
1. Data Loading     → transforms, DataLoader
2. Model Definition → nn.Module, Conv2d, Linear
3. Training Setup   → CrossEntropyLoss, Adam optimizer
4. Training Loop    → forward, backward, step
5. Evaluation       → model.eval(), torch.no_grad()
6. Save/Load        → state_dict
```

With ~5 epochs of training, you should get **~99% accuracy**!

## Phase 1 Complete!

You now understand:
- **Tensors**: The data structure for deep learning
- **Autograd**: Automatic gradient computation
- **nn.Module**: How to build neural networks
- **Training loop**: Forward → Loss → Backward → Update
- **Complete pipeline**: Data → Model → Train → Evaluate → Save

**Next: Phase 2** - Image Segmentation with U-Net.

The jump from classification (one label per image) to segmentation (one label per pixel) is where things get interesting for your spine/knee work!